# Notebook 1: Data Cleaning and Understanding

In questo notebook partiamo dal dataset grezzo (Bronze Layer) in formato Parquet. Dato che il dataset originale ci è stato fornito senza metadati (solo nomi di colonne), la prima fase è dedicata al **Data Understanding** (Reverse Engineering) per interpretare la struttura del dato, identificare le variabili chiave e isolare i valori sentinella.

Successivamente effettuiamo il **Data Cleaning** vero e proprio:
1. Deriviamo un identificatore univoco per partita (`match_id`).
2. Isoliamo gli eventi di gioco reale, filtrando i periodi `PreMatch` e `PostGame`.
3. Salviamo il dataset pulito (Silver Layer), che fungerà da input ottimizzato per l'estrazione delle feature.

## 0. Caricamento Dati (Bronze Layer)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

FILE_PATH = Path('../data/dataset.parquet')
df = pd.read_parquet(FILE_PATH)
print(f"Dataset caricato: {df.shape[0]:,} righe × {df.shape[1]} colonne")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 1. Data Understanding (Reverse Engineering)
Dal dataset non documentato abbiamo dedotto la seguente struttura fondamentale:
*   **`Unnamed: 0`**: Indice che si azzera a ogni nuova partita. Lo sfrutteremo per creare il `match_id`.
*   **`period_displayName`**: Indica il momento dell'evento. Abbiamo notato la presenza di periodi extra-gara da ignorare.
*   **`minute`**: Timestamp. Esplorando i dati, si notano dei valori sentinella (32767) associati ad anomalie o eventi di sistema.

In [ ]:
print("--- Periodi presenti ---")
print(df['period_displayName'].value_counts())

print("\n--- Valori sentinella in 'minute' ---")
sentinel = df[df['minute'] == 32767]
print(f"Righe con minute=32767: {len(sentinel)} → Appartengono ai periodi: {sentinel['period_displayName'].unique().tolist()}")

## 2. Data Cleaning
Ora che conosciamo le insidie, procediamo con la pulizia sistematica:

In [ ]:
# 1. Creazione del match_id
# Riordiniamo per l'indice originale per garantire la sequenzialità temporale
df = df.sort_index()
df['match_id'] = (df['Unnamed: 0'] == 0).cumsum()

n_matches = df['match_id'].nunique()
print(f"Partite identificate: {n_matches} (attese: 380 per PL 2020-21)")

# 2. Filtro eventi di gioco effettivo
# Escludiamo PreMatch e PostGame (che contengono anche i valori sentinella 32767)
df_clean = df[
    df['period_displayName'].isin(['FirstHalf', 'SecondHalf'])
].copy()

print(f"\nDimensioni dataset dopo la pulizia: {df_clean.shape[0]:,} righe × {df_clean.shape[1]} colonne")

## 3. Salvataggio del dataset pulito (Silver Layer)

In [ ]:
OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'dataset_clean.parquet'

df_clean.to_parquet(OUT_PATH, engine='pyarrow', compression='snappy')
print(f"Dataset pulito salvato correttamente in: {OUT_PATH}")